In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def e_power_exp(L_alt, eta, L_null):
    denom = np.mean(np.exp(-eta * L_null)) #expectation over the n samples of X_tilde
    E = np.exp(-eta * L_alt) / denom
    return np.mean(np.log(E)) #expectation over the alternative (n samples of X, Y, Z and different m_hat)


def e_power_lin(L_alt, lam, mu0):
    E = 1 + lam * (mu0 - L_alt) #mu0 is already the expectation of L under the null
    # avoid log of negative values
    if np.any(E <= 0):
        return -np.inf
    return np.mean(np.log(E)) #expectation over the alternative (n samples of X, Y, Z)


def optimize_exp(L_alt, L_null, eta_grid):
    vals = [e_power_exp(L_alt, eta, L_null) for eta in eta_grid]
    idx = np.argmax(vals)
    return eta_grid[idx], vals[idx]


def optimize_lin(L_alt, mu0, lam_grid):
    vals = [e_power_lin(L_alt, lam, mu0) for lam in lam_grid]
    idx = np.argmax(vals)
    return lam_grid[idx], vals[idx]

In [ ]:
def sample_xyz(n, B, gamma):
    X = np.random.uniform(0, 1, n)
    Z = np.random.uniform(0, 1, n)
    eps = np.random.uniform(-B, B, n)
    Y = X + eps + gamma * Z
    return X, Z, Y


def features(X, Z):
    return np.column_stack([np.ones_like(X), X, Z])


class OnlineLinearModel:
    def __init__(self):
        self.coef_ = np.zeros(3)

    def fit(self, X, Z, Y):
        X = np.asarray(X)
        Z = np.asarray(Z)
        Y = np.asarray(Y)
        self.coef_ = np.linalg.lstsq(features(X, Z), Y, rcond=None)[0]
        return self

    def predict(self, X, Z):
        X = np.asarray(X)
        Z = np.asarray(Z)
        return features(X, Z) @ self.coef_


def fit_linear_model(X, Z, Y):
    return OnlineLinearModel().fit(X, Z, Y)


def generate_data(n, B, gamma, pretrain_n=200):
    X_train, Z_train, Y_train = sample_xyz(pretrain_n, B, gamma)
    model = fit_linear_model(X_train, Z_train, Y_train)

    X, Z, Y = sample_xyz(n, B, gamma)
    X_tilde = np.random.uniform(0, 1, n)

    L_alt = np.empty(n)
    L_null = np.empty((n, n))

    X_history = np.empty(pretrain_n + n)
    Z_history = np.empty(pretrain_n + n)
    Y_history = np.empty(pretrain_n + n)
    X_history[:pretrain_n] = X_train
    Z_history[:pretrain_n] = Z_train
    Y_history[:pretrain_n] = Y_train

    for t in range(n):
        pred_alt = model.predict([X[t]], [Z[t]])[0]
        pred_null = model.predict(X_tilde, np.full(n, Z[t]))

        L_alt[t] = np.clip((Y[t] - pred_alt)**2, 0, 1)
        L_null[:, t] = np.clip((Y[t] - pred_null)**2, 0, 1)

        end = pretrain_n + t + 1
        X_history[end - 1] = X[t]
        Z_history[end - 1] = Z[t]
        Y_history[end - 1] = Y[t]
        model.fit(X_history[:end], Z_history[:end], Y_history[:end])

    return L_null, L_alt

In [ ]:
np.random.seed(0)
n = 500
pretrain_n = 50

eta_grid = np.linspace(0, 20, 400)
lam_grid = np.linspace(0, 1, 100)

noise_levels = np.linspace(0.01, 2, 50)

exp_powers = []
lin_powers = []

for noise in noise_levels:
    L_null, L_alt = generate_data(n, noise, 0.1, pretrain_n=pretrain_n)
    # print(L_null)
    mu0 = np.mean(L_null, axis = 0)
    # print(mu0)

    p_exp = optimize_exp(L_alt, L_null, eta_grid) #returns pair of optimal eta and optimal value
    p_lin = optimize_lin(L_alt, mu0, lam_grid) #returns pair of optimal lambda and optimal value

    exp_powers.append(p_exp)
    lin_powers.append(p_lin)

In [ ]:
plt.figure()
plt.plot(noise_levels, np.array(exp_powers)[:, 1], label="Exponential e-value")
plt.plot(noise_levels, np.array(lin_powers)[:, 1], label="Linear e-value")

plt.xlabel("Noise level")
plt.ylabel("Optimized e-power")
plt.title("Linear vs Exponential (bounded, correct construction)")
plt.legend()
plt.grid()

plt.show()
plt.savefig("icml_exp_1.pdf", bbox_inches="tight")

In [ ]:
plt.figure()
plt.plot(noise_levels[20:], np.array(exp_powers)[20:, 1], label="Exponential e-value")
plt.plot(noise_levels[20:], np.array(lin_powers)[20:, 1], label="Linear e-value")

plt.xlabel("Noise level")
plt.ylabel("Optimized e-power")
plt.title("Linear vs Exponential (bounded, correct construction)")
plt.legend()
plt.grid()

plt.show()
plt.savefig("icml_exp_2.pdf", bbox_inches="tight")

In [ ]:
def e_values_exp_per_timestep(L_alt, L_null, eta):
    denom = np.mean(np.exp(-eta * L_null), axis=0)
    return np.exp(-eta * L_alt) / denom


def e_values_lin_per_timestep(L_alt, mu0, lam):
    return 1 + lam * (mu0 - L_alt)


B_martingale = 0.9
L_null_B, L_alt_B = generate_data(n, B_martingale, 0.1, pretrain_n=pretrain_n)
mu0_B = np.mean(L_null_B, axis=0)

eta_B, _ = optimize_exp(L_alt_B, L_null_B, eta_grid)
lam_B, _ = optimize_lin(L_alt_B, mu0_B, lam_grid)

E_exp_B = e_values_exp_per_timestep(L_alt_B, L_null_B, eta_B)
E_lin_B = e_values_lin_per_timestep(L_alt_B, mu0_B, lam_B)

log_product_exp_B = np.cumsum(np.log(E_exp_B))
log_product_lin_B = np.cumsum(np.log(E_lin_B))
timesteps = np.arange(1, n + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(noise_levels, np.array(exp_powers)[:, 1], label="Exponential e-value")
axes[0].plot(noise_levels, np.array(lin_powers)[:, 1], label="Linear e-value")
axes[0].set_xlabel("Noise level")
axes[0].set_ylabel("Optimized e-power")
axes[0].set_title("Linear vs Exponential")
axes[0].legend()
axes[0].grid()

axes[1].plot(timesteps, log_product_exp_B, label="Exponential e-value")
axes[1].plot(timesteps, log_product_lin_B, label="Linear e-value")
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Log product")
axes[1].set_title(f"Cumulative log product, B = {B_martingale}")
axes[1].legend()
axes[1].grid()

plt.show()
fig.savefig("icml_exp_1_log_products.pdf", bbox_inches="tight")
